# Econ Causal Lab：从结果到研究问题

本笔记只读取仓库中的聚合结果，并运行一个合成数据小例子。无须下载真实微观数据。先在仓库安装 `python -m pip install -e .`。完整方法与研究边界见 `docs/methodology.md`。

In [1]:
import json
from pathlib import Path
import pandas as pd
root = Path.cwd() if (Path.cwd() / 'artifacts').exists() else Path.cwd().parent
report = json.loads((root / 'artifacts/results.json').read_text(encoding='utf-8'))
print(report['configuration'])

{'folds': 5, 'seeds': [42, 7, 2026], 'clip': 0.01, 'features': ['age', 'education', 'black', 'hispanic', 'married', 'nodegree', 're74', 're75'], 'outcome': 're78', 'treatment': 'treatment', 'primary_seed': 42, 'outcome_units': 'historical USD, as supplied; no present-value conversion', 'placebo': {'outcome': 're75', 'features': ['age', 'education', 'black', 'hispanic', 'married', 'nodegree', 're74'], 'seed': 42}, 'simulation': {'repetitions': 100, 'n': 1000, 'folds': 3, 'seed': 2026, 'clip': 0.01}}


## 1. 同样的受训者，为什么结论改变？

随机实验参考有抽样误差；观察性结果接近它也不是识别成立的证明。

In [2]:
table = pd.DataFrame(report['results'])
print(table[['dataset','method','estimate','ci_low','ci_high']].round(2).to_string(index=False))

      dataset method  estimate   ci_low  ci_high
 experimental  naive   1794.34   479.19  3109.50
 experimental linear   1854.61   508.20  3201.02
 experimental    hgb   2050.77   564.58  3536.97
observational  naive  -8497.52 -9641.04 -7353.99
observational linear   1231.97   -38.89  2502.83
observational    hgb   1757.42   199.44  3315.41


## 2. 培训能改变过去吗？

检查 `re75` 负对照。结果变量已从特征中移除。区间含零不证明无混杂，不含零则需要调查。

In [3]:
placebo = pd.DataFrame(report['placebo'])
print(placebo[['dataset','method','estimate','ci_low','ci_high']].round(2).to_string(index=False))

      dataset method  estimate    ci_low   ci_high
 experimental  naive    265.15   -332.74    863.03
 experimental linear    433.68    -38.23    905.58
 experimental    hgb     86.39   -427.90    600.69
observational  naive -12118.75 -12604.39 -11633.11
observational linear  -1337.76  -1863.08   -812.45
observational    hgb  -1212.16  -1802.36   -621.96


## 3. 已知真值时，区间有多可靠？

经验覆盖率来自有限次重复，不能认为 0/100 就是总体概率严格为零。

In [4]:
simulation = pd.DataFrame(report['simulation'])
print(simulation[['scenario','method','bias','rmse','coverage','repetitions']].round(3).to_string(index=False))

          scenario method     bias     rmse  coverage  repetitions
      good_overlap  naive 1335.623 1347.602      0.00          100
      good_overlap linear  565.452  587.237      0.06          100
      good_overlap    hgb   88.522  203.520      0.94          100
      weak_overlap  naive 2480.955 2485.837      0.00          100
      weak_overlap linear 1252.186 1286.590      0.01          100
      weak_overlap    hgb  531.832  609.528      0.49          100
hidden_confounding  naive 2707.581 2713.706      0.00          100
hidden_confounding linear 2163.861 2171.089      0.00          100
hidden_confounding    hgb 1816.049 1828.508      0.00          100


## 4. 亲手调用估计器

下面是另一份合成样本，真值 2000。一次估计偏离真值是正常现象；不能以挑种子代替重复实验。

In [5]:
from econ_causal_lab.simulation import generate
from econ_causal_lab import cross_fit, aipw_att
x, y, d, truth = generate(n=1000, seed=42)
fold_predictions = cross_fit(x, y, d, method='hgb', folds=3, seed=42)
result, influence = aipw_att(y, d, fold_predictions.propensity, fold_predictions.m0)
print('Known synthetic ATT:', truth)
print({key: round(value, 2) for key, value in result.items()})
print('Mean influence (should be near zero):', round(float(influence.mean()), 10))

Known synthetic ATT: 2000.0
{'estimate': 2083.63, 'se': 205.46, 'ci_low': 1680.92, 'ci_high': 2486.34}
Mean influence (should be near zero): -0.0


## 自己扩展

先提出假设并固定方案，例如加入二次项的线性基线或单侧 ATT 截断。保存全部结果，解释失败案例；将自己完成的扩展与本仓库提供的 AI 辅助实现区分开。